In [1]:
# Install once per fresh runtime
!pip install -q -U transformers accelerate PyMuPDF typing_extensions pillow opencv-python-headless

import os
import json
import cv2
import fitz
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

CACHE = Path("/workspace/hf")
os.environ.update({
    "HF_HOME": str(CACHE),
    "HF_HUB_CACHE": str(CACHE / "hub"),
    "TRANSFORMERS_CACHE": str(CACHE / "transformers"),
    "HF_HUB_DISABLE_XET": "1",
})

(CACHE / "hub").mkdir(parents=True, exist_ok=True)
(CACHE / "transformers").mkdir(parents=True, exist_ok=True)


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
PDF_PATH = "mmc2.pdf"
PAPER_ID = "mmc2"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
DPI = 350
PAGE_DIR = Path("rendered_pages")
CANDIDATE_DIR = Path("panel_candidates")
CACHE_DIR = Path("vlm_cache")
for d in [PAGE_DIR, CANDIDATE_DIR, CACHE_DIR]:
    d.mkdir(exist_ok=True)

In [3]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=2560 * 28 * 28,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print("Model loaded")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Model loaded


In [4]:
def render_pdf(pdf_path, out_dir=PAGE_DIR, dpi=DPI):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)
    scale = dpi / 72
    doc = fitz.open(pdf_path)
    page_records = []

    for page_idx, page in enumerate(doc):
        page_num = page_idx + 1
        pix = page.get_pixmap(
            matrix=fitz.Matrix(scale, scale),
            alpha=False,
        )

        image_path = out_dir / f"page_{page_num:03d}.png"
        pix.save(image_path)
        text = page.get_text()
        page_records.append({
            "page": page_num,
            "image_path": str(image_path),
            "text": text,
            "width_px": pix.width,
            "height_px": pix.height,
        })

    doc.close()
    return page_records

page_records = render_pdf(PDF_PATH)
print(f"Rendered {len(page_records)} pages")

Rendered 38 pages


In [5]:
import re

def normalize_ws(text):
    return re.sub(r"\s+", " ", text or "").strip()


def full_paper_text(page_records):
    return "\n\n".join(
        f"--- PAGE {r['page']} ---\n{r['text']}"
        for r in page_records
    )


FULL_TEXT = full_paper_text(page_records)


def nearby_page_text(page_records, page_num, radius=1, max_chars_per_page=2500):
    chunks = []

    for p in range(page_num - radius, page_num + radius + 1):
        if 1 <= p <= len(page_records):
            text = normalize_ws(page_records[p - 1]["text"])
            chunks.append(f"--- PAGE {p} ---\n{text[:max_chars_per_page]}")

    return "\n\n".join(chunks)


def extract_figure_context(full_text, max_chars=5000):
    """
    General figure/caption retriever. Works across papers using Figure/Fig/Supplementary patterns.
    """
    text = normalize_ws(full_text)

    patterns = [
        r"\bFigure\s+\d+",
        r"\bFig\.\s*\d+",
        r"\bFigure\s+S\d+",
        r"\bFig\.\s*S\d+",
        r"\bSupplementary\s+Figure\s+\d+",
        r"\bSupplementary\s+Fig\.\s*\d+",
        r"\bExtended\s+Data\s+Fig\.\s*\d+",
    ]

    matches = []
    for pat in patterns:
        matches.extend(re.finditer(pat, text, flags=re.IGNORECASE))

    chunks = []
    for m in matches:
        start = max(0, m.start() - 300)
        end = min(len(text), m.start() + 2500)
        chunks.append(text[start:end])

    # Deduplicate
    seen = set()
    deduped = []
    for c in chunks:
        key = c[:250].lower()
        if key not in seen:
            seen.add(key)
            deduped.append(c)

    return "\n\n".join(deduped)[:max_chars]


BIO_SAMPLE_KEYWORDS = [
    "cell", "cells", "cell line", "cell lines",
    "tissue", "tissues", "tumor", "tumour",
    "lysate", "lysates", "sample", "samples",
    "mouse", "mice", "rat", "human", "patient",
    "primary", "culture", "cultured",
    "organoid", "xenograft", "biopsy",
    "fibroblast", "macrophage", "neuron", "epithelial",
    "breast", "colon", "lung", "liver", "brain", "kidney",
    "western blot", "immunoblot", "immunoblotting",
]


def retrieve_relevant_text_snippets(full_text, query_terms, max_snippets=16, window=350):
    """
    Generic text retrieval for biological sample / methods context.
    No hardcoded cell lines.
    """
    text = normalize_ws(full_text)
    low = text.lower()

    snippets = []

    for term in query_terms:
        term_low = term.lower()
        start = 0

        while True:
            idx = low.find(term_low, start)
            if idx == -1:
                break

            s = max(0, idx - window)
            e = min(len(text), idx + len(term) + window)
            snippets.append(text[s:e])

            start = idx + len(term_low)

            if len(snippets) >= max_snippets * 3:
                break

    # Deduplicate
    seen = set()
    deduped = []
    for snip in snippets:
        key = snip[:250].lower()
        if key not in seen:
            seen.add(key)
            deduped.append(snip)

    return "\n\n".join(f"- {s}" for s in deduped[:max_snippets])


def build_text_context_for_candidate(rec, page_records, full_text, max_chars=12000):
    nearby = nearby_page_text(
        page_records,
        rec["page"],
        radius=1,
        max_chars_per_page=2500,
    )

    figure_context = extract_figure_context(
        full_text,
        max_chars=5000,
    )

    sample_methods_context = retrieve_relevant_text_snippets(
        full_text,
        BIO_SAMPLE_KEYWORDS,
        max_snippets=16,
        window=350,
    )

    context = f"""
PAPER_ID: {rec["paper_id"]}
CROP_PAGE: {rec["page"]}
CROP_BBOX_PAGE: {rec["bbox_page"]}

NEARBY_PAGE_TEXT:
{nearby}

LIKELY_FIGURE_OR_CAPTION_CONTEXT:
{figure_context}

PAPER_SAMPLE_OR_METHODS_SNIPPETS:
{sample_methods_context}
"""

    return context[:max_chars]

In [6]:
def expand_bbox(x, y, w, h, W, H, left=240, top=180, right=60, bottom=60):
    x0 = max(0, x - left)
    y0 = max(0, y - top)
    x1 = min(W, x + w + right)
    y1 = min(H, y + h + bottom)
    return x0, y0, x1, y1

def candidate_score(img_bgr):
    h, w = img_bgr.shape[:2]
    if w < 180 or h < 80:
        return 0.0

    area = w * h
    aspect = w / max(h, 1)
    
    if area < 40_000:
        return 0.0

    if aspect < 0.6 or aspect > 8.0:
        return 0.0

    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1].mean()
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    dark_frac = (gray < 210).mean()
    
    if dark_frac < 0.01 or dark_frac > 0.65:

        return 0.0

    binary = (gray < 210).astype(np.uint8) * 255
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 3))
    horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel)
    horizontal_frac = (horizontal > 0).mean()
    inv = 255 - gray
    row_profile = inv.mean(axis=1)
    col_profile = inv.mean(axis=0)
    row_signal = row_profile.std() / (row_profile.mean() + 1e-6)
    col_signal = col_profile.std() / (col_profile.mean() + 1e-6)
    score = 0.0
    score += max(0, 1 - sat / 80) * 0.25
    score += min(dark_frac / 0.18, 1) * 0.20
    score += min(horizontal_frac / 0.08, 1) * 0.30
    score += min(row_signal / 1.8, 1) * 0.15
    score += min(col_signal / 2.5, 1) * 0.10

    if sat > 70:
        score *= 0.4

    return float(score)

def generate_candidates(page_records, min_score=0.35):
    records = []

    for page_rec in page_records:
        page_num = page_rec["page"]
        page_path = page_rec["image_path"]
        img = cv2.imread(page_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        H, W = gray.shape
        mask = (gray < 245).astype(np.uint8) * 255
        # Bigger kernel = fewer, larger panel-ish crops.
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (45, 45))
        merged = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        merged = cv2.dilate(merged, kernel, iterations=1)
        contours, _ = cv2.findContours(
            merged,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        for c in contours:
            x, y, w, h = cv2.boundingRect(c)
            if w * h < 40_000:
                continue
            if w > 0.95 * W and h > 0.95 * H:
                continue
            tight_crop = img[y:y+h, x:x+w]
            score = candidate_score(tight_crop)
            if score < min_score:
                continue
            x0, y0, x1, y1 = expand_bbox(x, y, w, h, W, H)
            expanded_crop = img[y0:y1, x0:x1]
            out = CANDIDATE_DIR / f"page_{page_num:03d}_cand_{len(records):04d}.png"
            cv2.imwrite(str(out), expanded_crop)
            records.append({
                "paper_id": PAPER_ID,
                "page": page_num,
                "bbox_page": [int(x0), int(y0), int(x1), int(y1)],
                "tight_bbox_page": [int(x), int(y), int(x + w), int(y + h)],
                "candidate_path": str(out),
                "cv_score": score,
            })

    records.sort(key=lambda r: r["cv_score"], reverse=True)
    return records

candidates = generate_candidates(page_records, min_score=0.35)
print(f"{len(candidates)} candidates after CV filtering")
for r in candidates[:20]:
    print(r["page"], round(r["cv_score"], 3), r["candidate_path"])

240 candidates after CV filtering
30 0.878 panel_candidates/page_030_cand_0189.png
6 0.873 panel_candidates/page_006_cand_0046.png
6 0.869 panel_candidates/page_006_cand_0048.png
26 0.866 panel_candidates/page_026_cand_0170.png
24 0.866 panel_candidates/page_024_cand_0162.png
24 0.866 panel_candidates/page_024_cand_0158.png
24 0.866 panel_candidates/page_024_cand_0168.png
8 0.865 panel_candidates/page_008_cand_0059.png
24 0.865 panel_candidates/page_024_cand_0163.png
24 0.865 panel_candidates/page_024_cand_0157.png
24 0.864 panel_candidates/page_024_cand_0165.png
6 0.864 panel_candidates/page_006_cand_0049.png
24 0.864 panel_candidates/page_024_cand_0161.png
24 0.863 panel_candidates/page_024_cand_0166.png
8 0.862 panel_candidates/page_008_cand_0058.png
26 0.861 panel_candidates/page_026_cand_0172.png
6 0.861 panel_candidates/page_006_cand_0045.png
30 0.861 panel_candidates/page_030_cand_0196.png
6 0.86 panel_candidates/page_006_cand_0047.png
28 0.859 panel_candidates/page_028_cand_018

In [7]:
qwen_records = [
    r for r in candidates
    if r["cv_score"] >= 0.65
]
print(f"Sending {len(qwen_records)} candidates to Qwen")

Sending 84 candidates to Qwen


In [8]:
PROMPT = """
You are given ONE cropped candidate image from a scientific paper and paper text.

Return ONLY valid JSON. No markdown. No prose.

Use the IMAGE as the source of truth for:
- whether this is a western blot/immunoblot/protein band panel
- visible target labels
- visible lanes
- whether each band is present, absent, or uncertain

Use the PAPER TEXT only for:
- figure caption
- biological sample / cell line / tissue / organism
- treatment context
- abbreviation expansion

Do not use text to invent bands not visible in the image.
Use nearby page text, figure/caption context, and methods snippets to infer biological_sample, cell_line_tissue, organism, and sample_type. Prefer the caption/nearby page text over general methods text. If multiple samples are plausible, list them separated by "; " and add a warning.
If multiple samples are plausible, list them separated by "; " and add a warning.
If unclear, use "".

If this is not a western blot/immunoblot/protein band panel, return:
{"is_western_blot": false, "reason": "not a western blot"}

If it is a western blot/immunoblot/protein band panel, return:
{
  "is_western_blot": true,
  "panel_label": null,
  "figure_label": "",
  "figure_caption": "",
  "biological_sample": "",
  "cell_line_tissue": "",
  "organism": "",
  "sample_type": "",
  "treatment_context": "",
  "targets_top_to_bottom": [
    {"row_index": 1, "target": "", "is_loading_control": false, "confidence": "low"}
  ],
  "lanes_left_to_right": [
    {"lane_index": 1, "condition": "", "confidence": "low"}
  ],
  "bands": [
    {"row_index": 1, "target": "", "lane_index": 1, "band_state": "present", "confidence": "low"}
  ],
  "warnings": []
}

Rules:
- One band entry is required for every visible target row × every visible lane.
- band_state must be "present", "absent", or "uncertain".
- Use "uncertain" for faint, smeared, cropped, merged, or ambiguous bands.
- Do not estimate fold change or intensity.
- Read target names exactly as visible in the image.
- Loading controls include Actin, β-actin, GAPDH, Tubulin, HSP90, Vinculin, and total protein.
- If lane labels are shown as + / - grids, encode condition as readable text, e.g. "Drug +; siRNA -".
"""

In [9]:
def parse_json(raw):
    raw = raw.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(raw[start:end + 1])
        except json.JSONDecodeError:
            pass

    return {
        "error": "bad_json",
        "raw": raw[:1200],
    }

def load_for_vlm(path, max_side=1200):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))

    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)))
    return img

def page_text_for(page_num, max_chars=3500):
    text = page_records[page_num - 1]["text"]
    return text[:max_chars]

@torch.inference_mode()
def extract_with_qwen(image, local_text, max_new_tokens=1800):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT + "\n\n--- LOCAL TEXT ---\n" + local_text},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",

    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
    )

    raw = processor.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return parse_json(raw)

def cached_extract(rec):
    cache_path = CACHE_DIR / f"{Path(rec['candidate_path']).stem}.json"

    if cache_path.exists():
        return json.loads(cache_path.read_text())

    image = load_for_vlm(rec["candidate_path"], max_side=1200)

    local_text = build_text_context_for_candidate(
        rec=rec,
        page_records=page_records,
        full_text=FULL_TEXT,
        max_chars=12000,
    )

    data = extract_with_qwen(
        image,
        local_text=local_text,
        max_new_tokens=1800,
    )

    cache_path.write_text(json.dumps(data, indent=2))
    return data

In [10]:
OUTPUT_JSONL = Path("western_blot_extractions.jsonl")
OUTPUT_JSON = Path("western_blot_extractions.json")

# Resume from JSONL if it exists.
done_paths = set()
results = []
if OUTPUT_JSONL.exists():
    with OUTPUT_JSONL.open() as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                results.append(record)
                done_paths.add(record["candidate_path"])

with OUTPUT_JSONL.open("a") as f:
    for i, rec in enumerate(qwen_records):
        if rec["candidate_path"] in done_paths:
            continue

        print(
            f"{i+1}/{len(qwen_records)} "
            f"score={rec['cv_score']:.3f} "
            f"page={rec['page']} "
            f"{rec['candidate_path']}"
        )

        extraction = cached_extract(rec)

        output_record = {
            **rec,
            "extraction": extraction,
        }

        results.append(output_record)
        f.write(json.dumps(output_record) + "\n")
        f.flush()

# Final JSON array.
with OUTPUT_JSON.open("w") as f:
    json.dump(results, f, indent=2)

# positives-only JSON.
western_only = [
    r for r in results
    if isinstance(r.get("extraction"), dict)
    and r["extraction"].get("is_western_blot") is True
]

with open("western_blot_extractions_positive_only.json", "w") as f:
    json.dump(western_only, f, indent=2)

print("Done")
print(f"All candidate results: {len(results)} -> {OUTPUT_JSON}")
print(f"Western blot positives: {len(western_only)} -> western_blot_extractions_positive_only.json")

1/84 score=0.878 page=30 panel_candidates/page_030_cand_0189.png
2/84 score=0.873 page=6 panel_candidates/page_006_cand_0046.png
3/84 score=0.869 page=6 panel_candidates/page_006_cand_0048.png
4/84 score=0.866 page=26 panel_candidates/page_026_cand_0170.png
5/84 score=0.866 page=24 panel_candidates/page_024_cand_0162.png
6/84 score=0.866 page=24 panel_candidates/page_024_cand_0158.png
7/84 score=0.866 page=24 panel_candidates/page_024_cand_0168.png
8/84 score=0.865 page=8 panel_candidates/page_008_cand_0059.png
9/84 score=0.865 page=24 panel_candidates/page_024_cand_0163.png
10/84 score=0.865 page=24 panel_candidates/page_024_cand_0157.png
11/84 score=0.864 page=24 panel_candidates/page_024_cand_0165.png
12/84 score=0.864 page=6 panel_candidates/page_006_cand_0049.png
13/84 score=0.864 page=24 panel_candidates/page_024_cand_0161.png
14/84 score=0.863 page=24 panel_candidates/page_024_cand_0166.png
15/84 score=0.862 page=8 panel_candidates/page_008_cand_0058.png
16/84 score=0.861 page=2